# **LIMPIEZA Y TRANSFORMACIÓN DE LOS DATOS**

Este notebook aplica las transformaciones de limpieza derivadas del EDA y realiza validaciones básicas para asegurar consistencia y calidad de los datos antes del análisis.

**Principios**
- Transformar y exportar el dataset limpio a `data/processed/`.
- Documentar cada transformación y su motivación.

In [1]:
import pandas as pd
import numpy as np
import regex as re
import matplotlib.pyplot as plt
import seaborn as sns


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)

#### **1. CARGA DE LOS DATASETS**

- Importar los datasets originales que servirán como base para las fases de limpieza, corrección y transformación de los datos.
- Tras la carga, se realiza una verificación básica para confirmar la correcta lectura de los archivos, dimensiones y la estructura general.

##### **Descripción de los datasets**

Se cargan el dataset:

- **Dating App Reviews Dataset**  
  Dataset textual que contiene reviews de usuarios, junto con metadatos como rating, fecha y aplicación, orientado al análisis de sentimiento y evaluación de la experiencia de usuario.

In [2]:
path_df2 = "../data/raw/DatingAppReviewsDataset.csv"

df2 = pd.read_csv(path_df2,index_col=0)

print(f"Dataset 2: {df2.shape[0]} filas × {df2.shape[1]} columnas")
display(df2.head())

Dataset 2: 681994 filas × 6 columnas


,Name,Review,Rating,#ThumbsUp,Date&Time,App
0,linah sibanda,On this app i cant find a partner,5,0,18-02-2022 01:19,Tinder
1,Norman Johnson,Tinder would be so much better if we could spe...,3,0,18-02-2022 01:16,Tinder
2,David Hume,Still doesn't correctly notify matches or mess...,1,0,18-02-2022 01:11,Tinder
3,Last 1 Standing,"Got banned because I updated my bio to say ""I ...",2,0,18-02-2022 01:11,Tinder
4,Arthur Magamedov,Love it!,5,0,18-02-2022 01:06,Tinder


#### **2. HOMOGENEIZACIÓN EN EL NOMBRADO DE COLUMNAS**

- En esta sección se estandarizan los nombres de las columnas para garantizar consistencia y facilitar su uso en las fases posteriores del análisis.
- Primero, se normalizan los nombres al formato `snake_case`, eliminando espacios y caracteres especiales.
- Posteriormente, se renombran aquellas variables con nomenclatura ambigua para mejorar su claridad semántica (por ejemplo, `Date&Time` se renombra a `date_time`).
- Este proceso mejora la legibilidad del dataset y facilita su manipulación en el flujo de limpieza y análisis.

In [3]:
def normalizar_nombres_columnas(lista_columnas, mostrar_resumen=True):
    """
    Normaliza los nombres de columnas a formato snake_case estándar.

    Transformaciones aplicadas:
    - Eliminación de espacios al inicio y final
    - Conversión de CamelCase y PascalCase a snake_case
    - Sustitución de caracteres no alfanuméricos por guión bajo (_)
    - Conversión a minúsculas
    - Eliminación de guiones bajos duplicados
    - Eliminación de guiones bajos al inicio y final

    Parámetros
    ----------
    lista_columnas : list of str
        Lista de nombres de columnas originales.

    mostrar_resumen : bool, default=True
        Si True, muestra un resumen de la transformación realizada.

    Returns
    -------
    list of str
        Lista de nombres normalizados en formato snake_case.

    Ejemplo
    -------
    >>> normalizar_nombres_columnas(["Date&Time", "UserID", "App Name"])
    ['date_time', 'user_id', 'app_name']
    """

    nombres_normalizados = []

    for nombre in lista_columnas:

        # Eliminación de espacios extremos
        limpia = nombre.strip()

        # Conversión CamelCase / PascalCase → snake_case
        limpia = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', limpia)

        # Sustitución de caracteres no alfanuméricos por _
        limpia = re.sub(r'[^0-9a-zA-Z]+', '_', limpia)

        # Conversión a minúsculas
        limpia = limpia.lower()

        # Eliminación de guiones bajos duplicados
        limpia = re.sub(r'_+', '_', limpia)

        # Eliminación de guiones bajos en extremos
        limpia = limpia.strip('_')

        nombres_normalizados.append(limpia)

    if mostrar_resumen:
        print("Normalización de nombres de columnas completada.")
        print(f"Columnas procesadas: {len(lista_columnas)}")
        print("Transformaciones aplicadas:")
        for original, nuevo in zip(lista_columnas, nombres_normalizados):
            if original != nuevo:
                print(f"  {original} → {nuevo}")

    return nombres_normalizados

In [4]:
df2.columns = normalizar_nombres_columnas(df2.columns)

Normalización de nombres de columnas completada.
Columnas procesadas: 6
Transformaciones aplicadas:
  Name → name
  Review → review
  Rating → rating
  #ThumbsUp → thumbs_up
  Date&Time → date_time
  App → app


#### **3. LIMPIEZA DE VARIABLES CATEGÓRICAS**

- En esta sección se estandariza el formato de las variables categóricas.
- Se eliminan espacios al inicio y al final de los valores y se aplica un formato homogéneo de capitalización.
- Este proceso garantiza la consistencia de las categorías y evita la duplicación de valores equivalentes con distinta representación.

In [5]:
def limpiar_categoricas(df, columnas=None):
    """
    Limpia variables categóricas eliminando espacios y unificando el formato del texto.

    Aplica .str.strip().str.title() para garantizar consistencia en las categorías.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame a procesar.

    columnas : list, default=None
        Columnas a limpiar. Si es None, se aplicará a todas las columnas categóricas.

    Devuelve
    --------
    pandas.DataFrame
        DataFrame con las variables categóricas limpias.
    """

    # Si no se especifican columnas, detectar todas las categóricas
    if columnas is None:
        columnas = df.select_dtypes(include="O").columns

    # Aplicar limpieza básica
    for col in columnas:
        df[col] = df[col].str.strip().str.title()

    return df

In [6]:
df2 = limpiar_categoricas(df2)
df2.head()

,name,review,rating,thumbs_up,date_time,app
0,Linah Sibanda,On This App I Cant Find A Partner,5,0,18-02-2022 01:19,Tinder
1,Norman Johnson,Tinder Would Be So Much Better If We Could Spe...,3,0,18-02-2022 01:16,Tinder
2,David Hume,Still Doesn'T Correctly Notify Matches Or Mess...,1,0,18-02-2022 01:11,Tinder
3,Last 1 Standing,"Got Banned Because I Updated My Bio To Say ""I ...",2,0,18-02-2022 01:11,Tinder
4,Arthur Magamedov,Love It!,5,0,18-02-2022 01:06,Tinder


#### **4. TRATAMIENTO DE VALORES NULOS**

- En esta sección se gestionan los valores nulos identificados durante el EDA con el objetivo de garantizar la consistencia y calidad del dataset.  
- Se eliminan los registros que presentan valores nulos en variables categóricas o numéricas, incluyendo las reviews, ya que su ausencia impide su uso en el análisis de sentimiento y en el análisis estructural.  
- Este enfoque es apropiado dado el elevado volumen de datos, permitiendo depurar el dataset sin comprometer su representatividad.  
- Como resultado, se obtiene un dataset íntegro, preparado para las fases posteriores de análisis de sentimiento.

In [7]:
def tratamiento_nulos(df, mostrar_resumen = True):
    """
    Gestiona valores nulos eliminando filas con registros incompletos en variables
    categóricas y numéricas.

    Procedimiento:
    1) Identifica columnas categóricas (object/category) con nulos y elimina filas con nulos en ellas.
    2) Identifica columnas numéricas con nulos y elimina filas con nulos en ellas.
    3) Devuelve el DataFrame resultante (para actualizarlo por asignación).

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame de entrada.
    mostrar_resumen : bool, default=True
        Si True, imprime métricas del impacto del filtrado.

    Returns
    -------
    pandas.DataFrame
        DataFrame filtrado tras eliminar filas con nulos en columnas categóricas y numéricas.

    Nota
    ----
    Para “actualizar el dataframe”, reasigna:
    >>> df = tratamiento_nulos(df)
    """

    filas_iniciales = df.shape[0]

    # ---------- CATEGÓRICAS ----------
    cols_cat = df.select_dtypes(include=["object", "category"]).columns.tolist()
    cols_cat_nulos = [c for c in cols_cat if df[c].isna().any()]

    if cols_cat_nulos:
        filas_antes = df.shape[0]
        df = df.dropna(subset=cols_cat_nulos)
        filas_eliminadas_cat = filas_antes - df.shape[0]
    else:
        filas_eliminadas_cat = 0

    # ---------- NUMÉRICAS ----------
    cols_num = df.select_dtypes(include=["number"]).columns.tolist()
    cols_num_nulos = [c for c in cols_num if df[c].isna().any()]

    if cols_num_nulos:
        filas_antes = df.shape[0]
        df = df.dropna(subset=cols_num_nulos)
        filas_eliminadas_num = filas_antes - df.shape[0]
    else:
        filas_eliminadas_num = 0

    # ---------- RESUMEN ----------
    if mostrar_resumen:
        filas_finales = df.shape[0]
        print("Tratamiento de nulos completado")
        print(f"Filas iniciales: {filas_iniciales}")
        print(f"Filas eliminadas (categóricas): {filas_eliminadas_cat}")
        print(f"Filas eliminadas (numéricas): {filas_eliminadas_num}")
        print(f"Filas finales: {filas_finales}")
        print(f"Nulos restantes en el dataset: {df.isna().sum().sum()}")

        if cols_cat_nulos:
            print(f"Columnas categóricas con nulos consideradas: {cols_cat_nulos}")
        if cols_num_nulos:
            print(f"Columnas numéricas con nulos consideradas: {cols_num_nulos}")

    return df


In [8]:
df2 = tratamiento_nulos(df2)

Tratamiento de nulos completado
Filas iniciales: 681994
Filas eliminadas (categóricas): 1392
Filas eliminadas (numéricas): 0
Filas finales: 680602
Nulos restantes en el dataset: 0
Columnas categóricas con nulos consideradas: ['name', 'review']


#### **5. FILTRADO DE CALIDAD TEXTUAL EN REVIEWS**

- En esta sección se filtran las reviews para garantizar un nivel mínimo de calidad semántica antes del análisis de sentimiento.  
- Se eliminan aquellas entradas con contenido textual insuficiente, definido por un número mínimo de caracteres y palabras, ya que no aportan información relevante y pueden introducir ruido en el análisis.  
- Este proceso permite depurar el corpus textual, mejorando la fiabilidad de los resultados y la robustez de los análisis NLP posteriores.  
- Como resultado, se obtiene un conjunto de reviews informativas y adecuadas para el análisis de sentimiento y la evaluación de la experiencia del usuario.


In [9]:
def filtrar_reviews_por_calidad_textual(
    df,
    col_texto = "review",
    min_chars = 30,
    min_words = 5,
    mostrar_resumen = True
):
    """
    Filtra reviews con contenido textual insuficiente para garantizar calidad semántica
    en análisis de sentimiento y procesos NLP posteriores.

    Este procedimiento elimina registros cuyo contenido no cumple umbrales mínimos
    de longitud en caracteres y número de palabras, reduciendo ruido y mejorando
    la robustez analítica.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame de entrada que contiene la columna de reviews.

    col_texto : str, default="Review"
        Nombre de la columna que contiene el texto de las reviews.

    min_chars : int, default=30
        Número mínimo de caracteres requeridos para conservar una review.

    min_words : int, default=5
        Número mínimo de palabras requeridas para conservar una review.

    mostrar_resumen : bool, default=True
        Si True, muestra métricas de impacto del filtrado.

    Returns
    -------
    pd.DataFrame
        Nuevo DataFrame filtrado con únicamente reviews válidas.

    Notas
    -----
    Este filtrado mejora la calidad del corpus textual eliminando entradas no
    informativas como respuestas triviales o incompletas.

    Ejemplo
    -------
    >>> df_filtrado = filtrar_reviews_por_calidad_textual(df, "Review", 30, 5)
    """

    # Validación de existencia de la columna
    if col_texto not in df.columns:
        raise ValueError(f"La columna '{col_texto}' no existe en el DataFrame")

    # Registro de métricas iniciales
    filas_antes = df.shape[0]

    # Normalización del texto
    texto_normalizado = (
        df[col_texto]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    # Cálculo de métricas de calidad textual
    char_len = texto_normalizado.fillna("").str.len()
    word_count = texto_normalizado.fillna("").str.findall(r"\b\w+\b").str.len()

    # Construcción de máscara de filtrado
    mask_reviews_validas = (
        (char_len >= min_chars) &
        (word_count >= min_words)
    )

    # Aplicación del filtrado
    df_filtrado = df.loc[mask_reviews_validas].copy()

    # Métricas finales
    filas_despues = df_filtrado.shape[0]
    filas_eliminadas = filas_antes - filas_despues
    nulos_restantes = df_filtrado[col_texto].isna().sum()

    # Reporte de resultados
    if mostrar_resumen:
        print("Filtrado de calidad textual completado")
        print(f"Filas iniciales: {filas_antes}")
        print(f"Filas retenidas: {filas_despues}")
        print(f"Filas eliminadas por contenido insuficiente: {filas_eliminadas}")
        print(f"Nulos restantes en '{col_texto}': {nulos_restantes}")

    return df_filtrado


In [10]:
df2 = filtrar_reviews_por_calidad_textual(
    df2,
    col_texto="review",
    min_chars=30,
    min_words=5
)

Filtrado de calidad textual completado
Filas iniciales: 680602
Filas retenidas: 401630
Filas eliminadas por contenido insuficiente: 278972
Nulos restantes en 'review': 0


#### **6. PROCESAMIENTO DE VARIABLES TEMPORALES**

- En esta sección se convierte la variable temporal `date_time` al tipo datetime para garantizar su correcto tratamiento analítico.
- Esta transformación permite interpretar la variable como una dimensión temporal real, en lugar de texto, asegurando coherencia en ordenaciones y agregaciones.
- A partir de esta variable, se generan nuevas features temporales (`year`, `month`, `day`, `weekday`, `hour`) que facilitan el análisis de patrones de uso y evolución temporal de las reviews.
- Este procesamiento permite analizar tendencias, estacionalidad y comportamiento de los usuarios a lo largo del tiempo, mejorando la capacidad analítica del dataset.

In [11]:
def procesar_variable_temporal(
    df,
    col_fecha = "datetime",
    eliminar_original = False,
    mostrar_resumen = True
):
    """
    Convierte una columna a tipo datetime y genera variables temporales derivadas
    para facilitar el análisis temporal.

    Variables creadas:
    - year: año
    - month: mes (numérico)
    - month_name: nombre del mes
    - day: día del mes
    - weekday: nombre del día de la semana
    - hour: hora del día
    - date: fecha (sin hora)

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame de entrada.

    col_fecha : str, default="datetime"
        Nombre de la columna temporal a procesar.

    eliminar_original : bool, default=False
        Si True, elimina la columna original tras crear las variables derivadas.

    mostrar_resumen : bool, default=True
        Si True, muestra un resumen del proceso.

    Returns
    -------
    pandas.DataFrame
        DataFrame con la variable temporal convertida y nuevas features creadas.

    Ejemplo
    -------
    >>> df = procesar_variable_temporal(df, "date_time")
    """

    df = df.copy()

    # Validación
    if col_fecha not in df.columns:
        raise ValueError(f"La columna '{col_fecha}' no existe en el DataFrame")

    # Conversión a datetime
    df[col_fecha] = pd.to_datetime(df[col_fecha], errors="coerce")

    # Métricas
    nulos_generados = df[col_fecha].isna().sum()

    # Creación de variables derivadas
    df["year"] = df[col_fecha].dt.year
    df["month"] = df[col_fecha].dt.month
    df["month_name"] = df[col_fecha].dt.month_name()
    df["day"] = df[col_fecha].dt.day
    df["weekday"] = df[col_fecha].dt.day_name()
    df["hour"] = df[col_fecha].dt.hour
    df["date"] = df[col_fecha].dt.date

    # Eliminar original si se solicita
    if eliminar_original:
        df.drop(columns=[col_fecha], inplace=True)

    # Resumen
    if mostrar_resumen:
        print("Procesamiento de variable temporal completado")
        print(f"Columna procesada: {col_fecha}")
        print(f"Nulos generados tras conversión: {nulos_generados}")
        print("Variables creadas: year, month, month_name, day, weekday, hour, date")

    return df


In [12]:
df2 = procesar_variable_temporal(df2, col_fecha="date_time")

C:\Users\arant\AppData\Local\Temp\ipykernel_40696\3374885498.py:51: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col_fecha] = pd.to_datetime(df[col_fecha], errors="coerce")


Procesamiento de variable temporal completado
Columna procesada: date_time
Nulos generados tras conversión: 0
Variables creadas: year, month, month_name, day, weekday, hour, date


#### **7. REDUCCIÓN DEL DATASET MEDIANTE MUESTREO ESTRATIFICADO**

- Dado el elevado volumen del dataset original, se aplica un muestreo estratificado para reducir su tamaño a una muestra manejable sin comprometer su representatividad.
- El muestreo se realiza preservando la distribución conjunta de las variables `Rating` y `App`, consideradas dimensiones clave al estar directamente relacionadas con el sentimiento y el contexto de la review.
- Este enfoque garantiza que la muestra mantenga la misma estructura proporcional que el dataset original, evitando sesgos derivados de un muestreo aleatorio simple que podría infra o sobre-representar determinados perfiles.
- La muestra resultante permite reducir significativamente el coste computacional del análisis de sentimiento, manteniendo al mismo tiempo la validez analítica y la capacidad de generalización de los resultados.
- Este procedimiento optimiza el equilibrio entre eficiencia computacional y robustez estadística, facilitando el análisis sobre grandes volúmenes de datos en entornos con recursos limitados.

In [13]:
def muestreo_estratificado(df, n=50_000, cols_estrato=["rating", "app"], random_state=42):
    """
    Genera una muestra estratificada proporcional a partir de un DataFrame,
    preservando la distribución de las variables especificadas.

    Este procedimiento reduce el tamaño del dataset sin introducir sesgos
    relevantes, manteniendo la representatividad de variables clave como
    Rating y App.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame original.

    n : int, default=50_000
        Número de filas deseadas en la muestra.

    cols_estrato : list of str, default=["Rating", "App"]
        Columnas utilizadas para definir los estratos.

    random_state : int, default=42
        Semilla para reproducibilidad.

    Returns
    -------
    df_sample : pd.DataFrame
        DataFrame muestreado de forma estratificada.
    """

    df_temp = df.copy()
    df_temp.columns = df_temp.columns.str.strip()

    # resolver nombres reales de columnas (case-insensitive)
    col_map = {c.lower(): c for c in df_temp.columns}
    cols_resueltas = [col_map[c.lower()] for c in cols_estrato]

    # crear estrato
    df_temp["_stratum"] = df_temp[cols_resueltas].astype(str).agg("|".join, axis=1)

    # muestreo proporcional
    df_sample = (
        df_temp.groupby("_stratum", group_keys=False)
        .apply(lambda x: x.sample(
            n=max(1, round(n * len(x) / len(df_temp))),
            random_state=random_state
        ))
        .reset_index(drop=True)
    )

    # ajustar EXACTAMENTE a n filas
    if len(df_sample) > n:
        df_sample = df_sample.sample(n=n, random_state=random_state)
    elif len(df_sample) < n:
        faltan = n - len(df_sample)
        extra = df_temp.drop(df_sample.index).sample(n=faltan, random_state=random_state)
        df_sample = pd.concat([df_sample, extra], ignore_index=True)

    return df_sample.drop(columns="_stratum")

In [14]:
df2 = muestreo_estratificado(df2, n=50_000)

print(df2.shape)

(50000, 13)


C:\Users\arant\AppData\Local\Temp\ipykernel_40696\1092029279.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


#### **8. EXPORTACIÓN DEL DATASET LIMPIO**

- En esta sección se exportan los datasets tras aplicar los procesos de limpieza, tratamiento de nulos y transformación de variables.
- Los datasets se guardan en formato `.csv` dentro del directorio `data/processed/`, siguiendo una estructura organizada del proyecto.
- Esta exportación garantiza la reutilización de los mismos sin necesidad de repetir las fases de preparación.

In [15]:
output_path_reviews = "../data/processed/dating_app_reviews_clean.csv"

df2.to_csv(output_path_reviews, index=False)

print("Exportación completada correctamente:")
print(f"  - Reviews:        {output_path_reviews}")

Exportación completada correctamente:
  - Reviews:        ../data/processed/dating_app_reviews_clean.csv
